# Blackberry 3DGS on Google Colab (`engines_cuda`)

Uses **PyTorch + gsplat** (CUDA). Branch: **`feature/colab-train`**.

**Runtime -> GPU** required (T4/L4/A100 — not TPU).

### Important (2026-10-28)
The first MCMC run (`cuda_out_mcmc`, `splat_07000.ply`) **diverged**: Gaussian means
flew to ±1e5 while opacity collapsed. Do **not** `--resume` that folder.

This notebook defaults to a **fresh OUT** and a **safer** recipe:
1. Warmup with `--strategy default`, or
2. MCMC with much lower `--noise-lr` (default 5e5 is too violent on this COLMAP scale).

Mesh: `engines_cuda/export/mesh_from_ckpt.py` (Open3D / PyTorch3D) — section 6.


## 1. Comparison configuration + GPU check

Start with a **fresh GPU runtime**. Prefer T4: like the local GTX 1650 it has compute capability 7.5, but more VRAM. This compares the same PyTorch/CUDA/gsplat versions on Linux/GCC, **not** Windows/MSVC. The NVIDIA driver remains managed by Colab.

The Colab kernel can use a different Python version. A separate **Python 3.12.9** is downloaded with uv for the training environment. No driver, system Python or notebook kernel is replaced.


In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

PYTHON_VERSION = "3.12.9"
TORCH_VERSION = "2.15.0+cu134"
CUDA_TOOLKIT_VERSION = "13.4.3"  # NVIDIA package set containing nvcc 13.4.92
NVCC_VERSION = "13.4.92"
GSPLAT_COMMIT = "2a29ad06ba008bb34c9b942e82c2d46d39a194f4"
NINJA_VERSION = "1.13.2"
REPO = "/content/splats"
BRANCH = "feature/colab-train"
VENV = Path("/content/splats-cu134-env")
PYTHON = str(VENV / "bin" / "python")

if sys.platform != "linux":
    raise RuntimeError(f"This Colab comparison requires Linux; got {platform.platform()}")
print("Colab kernel Python:", sys.version, "| comparison Python:", PYTHON_VERSION, flush=True)
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["g++", "--version"], check=True)


## 2. Clone repo + install the pinned comparison stack

Uses the **actually installed local gsplat commit**, not a moving `main` or the previously pinned `512d366`. Installs NVIDIA's compiler/runtime/CCCL wheels without changing Colab's driver. All package and training commands use the isolated interpreter; no kernel restart is needed.

`BUILD_NO_CUDA=1` defers compilation to section 2b. `MAX_JOBS=1` limits host RAM use; the local default was 10. PyTorch3D is optional and is deliberately excluded from this build comparison (section 6).


In [ ]:
BOOTSTRAP = Path("/content/splats-bootstrap")
subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(BOOTSTRAP), "uv==0.13.6"], check=True)
UV = str(BOOTSTRAP / "bin" / "uv")
subprocess.run([UV, "python", "install", PYTHON_VERSION], check=True)
BASE_PYTHON = subprocess.check_output(
    [UV, "python", "find", "--managed-python", PYTHON_VERSION], text=True,
).strip()
actual_version = subprocess.check_output(
    [BASE_PYTHON, "-c", "import platform; print(platform.python_version())"], text=True,
).strip()
if actual_version != PYTHON_VERSION:
    raise RuntimeError(f"Expected comparison Python {PYTHON_VERSION}, got {actual_version}")
print("Comparison Python:", actual_version, "at", BASE_PYTHON, flush=True)


In [ ]:
if not Path(REPO).is_dir():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                    "https://github.com/aidazsolt-bot/splats.git", REPO], check=True)
else:
    for args in (["fetch", "origin", BRANCH], ["checkout", BRANCH], ["pull", "--ff-only"]):
        subprocess.run(["git", *args], cwd=REPO, check=True)
os.chdir(REPO)

subprocess.run([BASE_PYTHON, "-m", "venv", "--without-pip", str(VENV)], check=True)
PIP = [sys.executable, "-m", "pip", "--python", PYTHON]
os.environ["BUILD_NO_CUDA"] = "1"
os.environ["MAX_JOBS"] = "1"
os.environ["VERBOSE"] = "1"
os.environ["NINJA_STATUS"] = "[%f/%t %es] "
subprocess.run([*PIP, "install", f"torch=={TORCH_VERSION}",
                "--index-url", "https://download.pytorch.org/whl/cu134"], check=True)
subprocess.run([*PIP, "install", f"cuda-toolkit[nvcc,cudart,cccl]=={CUDA_TOOLKIT_VERSION}",
                f"ninja=={NINJA_VERSION}", "setuptools", "wheel"], check=True)
CONSTRAINTS = VENV / "comparison-constraints.txt"
CONSTRAINTS.write_text(f"torch=={TORCH_VERSION}\nninja=={NINJA_VERSION}\n", encoding="utf-8")
subprocess.run([*PIP, "install", "-c", str(CONSTRAINTS), "-r", "engines_cuda/requirements.txt"], check=True)
subprocess.run([*PIP, "install", "-c", str(CONSTRAINTS), "--no-build-isolation", "--force-reinstall", "--no-deps",
                f"gsplat @ git+https://github.com/nerfstudio-project/gsplat.git@{GSPLAT_COMMIT}"], check=True)
# Resolve gsplat dependencies without rebuilding or changing its pinned source.
subprocess.run([*PIP, "install", "-c", str(CONSTRAINTS), "gsplat==1.6.0"], check=True)
subprocess.run([*PIP, "check"], check=True)

CUDA_HOME = subprocess.check_output(
    [PYTHON, "-c", "import importlib.metadata as m; print(m.distribution('nvidia-cuda-nvcc').locate_file('nvidia/cu13'))"],
    text=True,
).strip()
cuda_root = Path(CUDA_HOME)
for required in ("bin/nvcc", "include/cuda_runtime.h", "include/cccl/cub/cub.cuh", "nvvm/libdevice/libdevice.10.bc", "lib/libcudart.so.13"):
    if not (cuda_root / required).is_file():
        raise RuntimeError(f"Incomplete CUDA toolkit: missing {cuda_root / required}")
# NVIDIA's runtime wheel provides the versioned library, but -lcudart needs this name.
if not (cuda_root / "lib/libcudart.so").exists():
    (cuda_root / "lib/libcudart.so").symlink_to("libcudart.so.13")
os.environ["CUDA_HOME"] = CUDA_HOME
os.environ["CUDA_PATH"] = CUDA_HOME
os.environ["PATH"] = f"{VENV / 'bin'}:{cuda_root / 'bin'}:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{cuda_root / 'lib'}:" + os.environ.get("LD_LIBRARY_PATH", "")
nvcc_output = subprocess.check_output([str(cuda_root / "bin/nvcc"), "--version"], text=True)
print(nvcc_output, flush=True)
if f"V{NVCC_VERSION}" not in nvcc_output:
    raise RuntimeError(f"Expected nvcc {NVCC_VERSION}, not the Colab default toolkit")
os.environ["GSPLAT_EXPECTED_PYTHON"] = PYTHON_VERSION
os.environ["GSPLAT_EXPECTED_COMMIT"] = GSPLAT_COMMIT
os.environ["GSPLAT_EXPECTED_TORCH"] = TORCH_VERSION
os.environ["GSPLAT_EXPECTED_NINJA"] = NINJA_VERSION


## 2a. Verify versions + basic CUDA operation (no gsplat build)

A failure here is an environment/driver/PyTorch problem, before gsplat compilation. This checks an actual GPU operation, not just device detection. The driver must support this PyTorch CUDA build; the notebook does not upgrade it or silently substitute another stack.


In [ ]:
verify_script = r'''
import importlib.metadata as m
import json
import os
import platform
import torch

print('Python:', platform.python_version(), '| local reference: 3.12.9', flush=True)
print('PyTorch:', torch.__version__, '| CUDA runtime:', torch.version.cuda, flush=True)
assert torch.__version__ == os.environ['GSPLAT_EXPECTED_TORCH']
assert torch.version.cuda == '13.4'
dist = m.distribution('gsplat')
source = json.loads(dist.read_text('direct_url.json'))
print('gsplat:', dist.version, '| source:', source, flush=True)
assert dist.version == '1.6.0'
assert source['vcs_info']['commit_id'] == os.environ['GSPLAT_EXPECTED_COMMIT']
assert m.version('ninja') == os.environ['GSPLAT_EXPECTED_NINJA']
assert torch.cuda.is_available(), 'CUDA unavailable: check the Colab GPU runtime and driver'
print('GPU:', torch.cuda.get_device_name(0), '| capability:', torch.cuda.get_device_capability(0),
      '| VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), flush=True)
print('PyTorch architectures:', torch.cuda.get_arch_list(), flush=True)
x = torch.randn(512, 512, device='cuda')
y = x @ x
assert torch.isfinite(y).all().item()
torch.cuda.synchronize()
print('PASS: basic PyTorch CUDA computation', flush=True)
'''
subprocess.run([PYTHON, "-u", "-c", verify_script], check=True)
capability = subprocess.check_output(
    [PYTHON, "-c", "import torch; print('.'.join(map(str, torch.cuda.get_device_capability(0))))"], text=True,
).strip()
os.environ["TORCH_CUDA_ARCH_LIST"] = capability
os.environ["TORCH_EXTENSIONS_DIR"] = f"/content/torch_extensions/cu134-{GSPLAT_COMMIT}-sm{capability.replace('.', '')}"
print("Build architecture:", capability, "| MAX_JOBS:", os.environ["MAX_JOBS"], flush=True)


## 2b. Build/load gsplat only (verbose, one compiler job)

This is primarily a **CPU / host RAM** test, not a GPU training benchmark. Compiler commands and Ninja progress stay visible. Python's standard-library `time` and `resource` modules report elapsed time, the Python process peak RSS and the largest reaped child-process peak RSS, including on normal build exceptions. These RSS values are **not combined process-tree memory**. No external `time` executable is needed. A successful rerun may reuse the extension cache and is not a cold-build timing.

Keep the final compiler error and resource report if it fails. A Linux success with the same versions narrows the issue toward the Windows/MSVC build path; it does not by itself prove a hardware fault or compare equal RAM limits.


In [ ]:
build_script = r'''
import resource
import time

started = time.perf_counter()
try:
    from gsplat.cuda._backend import _C
    assert _C is not None, 'gsplat CUDA extension unavailable'
    print('PASS: gsplat build/load', flush=True)
finally:
    elapsed = time.perf_counter() - started
    own = resource.getrusage(resource.RUSAGE_SELF)
    children = resource.getrusage(resource.RUSAGE_CHILDREN)
    print(f'Build/load elapsed: {elapsed:.1f} s', flush=True)
    print(f'Python peak RSS: {own.ru_maxrss / 1024:.1f} MiB', flush=True)
    print(f'Largest reaped child peak RSS: {children.ru_maxrss / 1024:.1f} MiB (not combined process-tree memory)', flush=True)
'''
subprocess.run([PYTHON, "-u", "-c", build_script], check=True)


## 2c. Tiny gsplat forward/backward GPU test

Only after the build passes: render one Gaussian and check output shapes and finite gradients. This isolates kernel execution from dataset loading and full-training memory use.


In [ ]:
raster_script = r'''
import torch
from gsplat import rasterization

means = torch.tensor([[0., 0., 3.]], device='cuda', requires_grad=True)
quats = torch.tensor([[1., 0., 0., 0.]], device='cuda', requires_grad=True)
scales = torch.full((1, 3), 0.1, device='cuda', requires_grad=True)
opacities = torch.tensor([0.8], device='cuda', requires_grad=True)
colors = torch.tensor([[1., 0.5, 0.2]], device='cuda', requires_grad=True)
viewmats = torch.eye(4, device='cuda')[None]
Ks = torch.tensor([[[32., 0., 16.], [0., 32., 16.], [0., 0., 1.]]], device='cuda')
image, alpha, _ = rasterization(means, quats, scales, opacities, colors, viewmats, Ks,
                                 width=32, height=32, render_mode='RGB')
assert image.shape == (1, 32, 32, 3) and alpha.shape == (1, 32, 32, 1)
assert torch.isfinite(image).all().item() and torch.isfinite(alpha).all().item()
assert alpha.max().item() > 0, 'Gaussian was not rendered'
(image.square().mean() + alpha.mean()).backward()
for name, value in [('means', means), ('quats', quats), ('scales', scales), ('opacities', opacities), ('colors', colors)]:
    assert value.grad is not None and torch.isfinite(value.grad).all().item(), f'Invalid gradient: {name}'
torch.cuda.synchronize()
print('PASS: gsplat forward/backward | peak GPU MiB:', round(torch.cuda.max_memory_allocated() / 2**20, 1), flush=True)
'''
subprocess.run([PYTHON, "-u", "-c", raster_script], check=True)


## 3. Mount Drive + dataset / dataset paths

Expected:
```
MyDrive/blackberry-splats/dataset/{cameras,images,points3D}.txt + images/
```

`OUT` must be a **new** folder after the failed MCMC run.


In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/blackberry-splats/dataset"
# NEW out dir — do not reuse .../cuda_out_mcmc (diverged means)
OUT = "/content/drive/MyDrive/blackberry-splats/cuda_out_stable"

os.makedirs(os.path.dirname(DATA), exist_ok=True)
if not os.path.isfile(f"{DATA}/cameras.txt"):
    print("Dataset missing — downloading blackberry.zip once...")
    os.makedirs(DATA, exist_ok=True)
    !wget -O /content/blackberry.zip \
        "https://github.com/danybittel/splats/releases/download/berries/blackberry.zip"
    !unzip -q -o /content/blackberry.zip -d "{DATA}"
else:
    print("Dataset already on Drive — skip zip download")

assert os.path.isfile(f"{DATA}/cameras.txt"), f"Missing COLMAP at {DATA}"
assert os.path.isdir(f"{DATA}/images"), f"Missing images/ at {DATA}"
os.makedirs(OUT, exist_ok=True)
print("DATA", DATA)
print("OUT", OUT)
!ls "{DATA}" | head


## 4. Smoke train (optional)

Stack check only. Uses `default` + `data-factor 4`.


In [ ]:
subprocess.run([PYTHON, "-u", "engines_cuda/run_train.py", "--smoke", "--data", DATA,
                "--out", f"{OUT}/smoke", "--data-factor", "4", "--strategy", "default"], cwd=REPO, check=True)


## 5. Full train (stable recipe)

Failed run used MCMC + default `--noise-lr 5e5` → means exploded by step 7000.

| Mode | When |
|------|------|
| `default` | Safer first full train (recommended) |
| `mcmc` | Only with **low** `NOISE_LR` (e.g. `5e3`…`5e4`) and fresh `OUT` |

| GPU | `DATA_FACTOR` |
|-----|----------------|
| T4 | start `2`, try `1` if VRAM allows |
| L4 / A100 | `1` |

After a **healthy** disconnect: `RESUME = True` and **same** `OUT` + flags.
Never resume the old `cuda_out_mcmc`.


In [ ]:
RESUME = False  # True only after a healthy disconnect on THIS OUT
STRATEGY = "default"  # or "mcmc" with low NOISE_LR
DATA_FACTOR = 2  # T4-safe; set 1 on L4/A100
CAP_MAX = 200000  # used by mcmc
MAX_STEPS = 30000  # raise to 45000 once stable
NOISE_LR = 5e3  # MCMC only; 5e5 diverged on this scene scale
ANTIALIASED = True

cmd = [PYTHON, "-u", "engines_cuda/run_train.py", "--data", DATA, "--out", OUT,
       "--strategy", STRATEGY, "--max-steps", str(MAX_STEPS), "--data-factor", str(DATA_FACTOR)]
if STRATEGY == "mcmc":
    cmd += ["--cap-max", str(CAP_MAX), "--noise-lr", str(NOISE_LR)]
if ANTIALIASED:
    cmd += ["--antialiased"]
if RESUME:
    cmd += ["--resume"]
print(cmd, flush=True)
subprocess.run(cmd, cwd=REPO, check=True)


## 5b. Quick PLY sanity check (after first `splat_*.ply`)

Extent should be ~**20–70** per axis (COLMAP scale), not 1e5+.
Opacity median should not sit near ~0.01 with most mass outside the scene.


In [ ]:
import numpy as np
from pathlib import Path

ply = sorted(Path(OUT).glob("splat_*.ply"))
assert ply, f"No splat_*.ply yet in {OUT}"
path = ply[-1]
raw = path.read_bytes()
i = raw.find(b"end_header\n")
hdr, body = raw[:i].decode(), raw[i + 11 :]
n = int([ln for ln in hdr.splitlines() if ln.startswith("element vertex")][0].split()[-1])
props = [ln.split()[-1] for ln in hdr.splitlines() if ln.startswith("property float")]
arr = np.frombuffer(body, dtype="<f4", count=n * len(props)).reshape(n, len(props))
ix = {p: j for j, p in enumerate(props)}
xyz = arr[:, [ix["x"], ix["y"], ix["z"]]]
opa = 1 / (1 + np.exp(-np.clip(arr[:, ix["opacity"]], -40, 40)))
extent = xyz.max(0) - xyz.min(0)
print(path.name, "n", n)
print("extent", extent)
print("center", xyz.mean(0))
print("opacity p50/mean", float(np.median(opa)), float(opa.mean()))
if extent.max() > 500:
    print("FAIL: means likely diverged — stop and change STRATEGY/NOISE_LR/OUT")
else:
    print("OK: extent looks COLMAP-scale")


## 6. List outputs / mesh from checkpoint

Open3D Poisson is available by default. Optional PyTorch3D cubify has its own source build and is **not part of the gsplat comparison**. Set `INSTALL_PYTORCH3D = True` below only when needed; its dependencies remain constrained to the comparison PyTorch/Ninja versions.


In [ ]:
subprocess.run(["ls", "-lh", OUT], check=True)
INSTALL_PYTORCH3D = False
if INSTALL_PYTORCH3D:
    subprocess.run([*PIP, "install", "-c", str(CONSTRAINTS), "--no-build-isolation",
                    "-r", "engines_cuda/requirements-pytorch3d.txt"], cwd=REPO, check=True)
# subprocess.run([PYTHON, "engines_cuda/export/mesh_from_ckpt.py", "--ckpt", f"{OUT}/ckpt_30000.pt",
#                 "--out", f"{OUT}/mesh_from_splat.ply"], cwd=REPO, check=True)
# For cubify, add "--backend", "pytorch3d", "--grid-size", "128" to the mesh command.
